# Notebook 2: When the Compiler Isn't Enough
## Plain BaseAnalogCkt subclasses: Neural ODE, Flow, Diffusion

This notebook demonstrates **Path B**: hand-written `BaseAnalogCkt` subclasses for architectures that Ark's `OptCompiler` cannot (yet) express directly.

**Why these need plain classes:**
- **Neural ODE / Flow**: a time-augmented input `[z, t]` is fed into an MLP. Ark's static CDG grammar has no notion of concatenating a time scalar into the state vector at runtime.
- **Diffusion (CLD SDE)**: second-order stochastic dynamics with Johnson-Nyquist thermal noise. Requires `is_stochastic=True`, noise coupled to `v` only, and a time-varying beta schedule - all awkward in a static grammar.

The compiler is powerful but not universal. Rather than force-fit every architecture through it, we drop to hand-written subclasses when the physics is simpler to express directly. The `BaseAnalogCkt` contract (`ode_fn`, `make_args`, `noise_fn`, `readout`) is unchanged, so gradient-based mismatch-aware retraining still works.


# Colab Setup

Run the code cell below once on a fresh Colab runtime. It will:
1. **Detect** the `neuro-analog` repository (walk-up, Colab paths, or uploaded zip).
2. **Clone** from GitHub if nothing is found.
3. **Install** system deps (`graphviz`, `z3-solver`) and Python packages (`torch`, `jax[cuda12]`, `diffrax`, ...).
4. **Clone & install** Ark from `WangYuNeng/Ark`.
5. **Verify** checkpoint directory and JAX device visibility.

Expected runtime: ~2 min on a warm GPU runtime, ~5 min cold.


In [ ]:
"""Canonical bootstrap for revised_ark_bridge notebooks.

On a fresh Colab runtime this installs dependencies and clones the repo.
If you already have the repo checked out locally (e.g. running in Jupyter),
it detects it, adds it to sys.path, and skips all installation.
"""

import sys, os, subprocess, importlib, zipfile, shutil

repo_name = "neuro-analog"
repo_path = None

# ── FAST PATH: already inside a local checkout? ──
# Walk up from cwd looking for neuro-analog/
cwd = os.getcwd()
parts = cwd.replace("\\", "/").split("/")
for i in range(len(parts), 0, -1):
    candidate = "/".join(parts[:i])
    if candidate.endswith(repo_name) and os.path.isdir(os.path.join(candidate, "neuro_analog")):
        repo_path = candidate
        break

# Validate and short-circuit if we already have the repo locally
if repo_path is not None:
    if not os.path.isdir(os.path.join(repo_path, "neuro_analog", "revised_ark_bridge")):
        print(f"WARNING: Found repo at {repo_path} but missing revised_ark_bridge. Ignoring.")
        repo_path = None
    else:
        if repo_path not in sys.path:
            sys.path.insert(0, repo_path)
        ckpt_root = os.path.join(repo_path, "experiments", "cross_arch_tolerance", "checkpoints")
        if not os.path.isdir(ckpt_root):
            print(f"WARNING: Checkpoints directory not found: {ckpt_root}. Proceeding with full setup...")
            repo_path = None
        else:
            # Verify Ark is reachable before short-circuiting
            try:
                import ark
            except ImportError:
                print(f"WARNING: repo found at {repo_path} but Ark is not importable.")
                print("Install Ark locally (e.g. pip install -e /path/to/Ark) and re-run.")
                raise SystemExit(1)
            print(f"Local repo detected at {repo_path} — skipping all installs.")
            print("Checkpoints path:", ckpt_root)
            print("\nSUCCESS! Setup complete.")
            raise SystemExit(0)

# ── SLOW PATH: fresh runtime / missing deps ──
# Force-upgrade jax + torch atomically so pip resolves a CUDA-compatible pair
subprocess.run(["pip", "install", "-q", "--upgrade", "jax[cuda12]", "torch"], check=False, capture_output=True, text=True)

# 1. Walk up from cwd looking for neuro-analog/
cwd = os.getcwd()
parts = cwd.replace("\\", "/").split("/")
for i in range(len(parts), 0, -1):
    candidate = "/".join(parts[:i])
    if candidate.endswith(repo_name) and os.path.isdir(os.path.join(candidate, "neuro_analog")):
        repo_path = candidate
        break

# 2. Check known Colab paths
if repo_path is None:
    for colab_path in [f"/content/{repo_name}", f"/content/real_new_content/{repo_name}"]:
        if os.path.isdir(os.path.join(colab_path, "neuro_analog")):
            repo_path = colab_path
            break

# 3. Check for uploaded zip files
if repo_path is None:
    for zip_dir in ["/content", "/content/real_new_content"]:
        zip_path = os.path.join(zip_dir, f"{repo_name}.zip")
        if os.path.exists(zip_path):
            extract_to = os.path.join(zip_dir, repo_name)
            if os.path.exists(extract_to):
                shutil.rmtree(extract_to)
            os.makedirs(extract_to, exist_ok=True)
            if not os.path.isfile(zip_path):
                print(f"Skipping {zip_path}: not a file (may be a directory).")
                continue
            file_size = os.path.getsize(zip_path)
            if file_size == 0:
                print(f"Skipping {zip_path}: file is empty.")
                continue
            print(f"Found zip at {zip_path} ({file_size} bytes). Extracting...")
            try:
                with zipfile.ZipFile(zip_path, 'r') as z:
                    for info in z.infolist():
                        # Windows backslashes in zip filenames must become Linux forward slashes
                        info.filename = info.filename.replace("\\", "/")
                        z.extract(info, extract_to)
            except zipfile.BadZipFile as e:
                print(f"Bad zip file at {zip_path}: {e}. Trying next location or git clone...")
                continue
            # Handle flat extraction (zip contains neuro_analog/ directly)
            if not os.path.isdir(os.path.join(extract_to, "neuro_analog")):
                flat_items = [d for d in os.listdir(extract_to) if os.path.isdir(os.path.join(extract_to, d))]
                if len(flat_items) == 1:
                    nested = os.path.join(extract_to, flat_items[0])
                    for item in os.listdir(nested):
                        shutil.move(os.path.join(nested, item), os.path.join(extract_to, item))
                    shutil.rmtree(nested)
            repo_path = extract_to
            print(f"Extracted zip to {repo_path}")
            break

# 4. Fallback: git clone
if repo_path is None:
    clone_target = f"/content/{repo_name}"
    print("Repository not found. Cloning from GitHub...")
    subprocess.run(
        ["git", "clone", "https://github.com/apumutyala/neuro-analog.git", clone_target],
        check=True, capture_output=True, text=True
    )
    repo_path = clone_target
    print(f"Cloned to {repo_path}")
else:
    print(f"Found existing repo at: {repo_path} (will validate and install missing deps)")

# Validate repo structure before accepting it
if repo_path is not None and not os.path.isdir(os.path.join(repo_path, "neuro_analog", "revised_ark_bridge")):
    print(f"WARNING: Found repo at {repo_path} but missing neuro_analog/revised_ark_bridge. Ignoring.")
    repo_path = None

# Add repo root to path
if repo_path is not None and repo_path not in sys.path:
    sys.path.insert(0, repo_path)

# --- Install system dependencies for Ark ---
try:
    import graphviz
except ImportError:
    print("Installing graphviz system package...")
    subprocess.run(["apt-get", "install", "-y", "-qq", "graphviz", "libgraphviz-dev"],
                   check=False, capture_output=True, text=True)

try:
    import z3
except ImportError:
    print("Installing z3-solver...")
    subprocess.run(["pip", "install", "-q", "z3-solver"],
                   check=True, capture_output=True, text=True)

# --- Install torch FIRST (Ark README says torch before jax due to cudnn) ---
try:
    import torch
    print("torch already available.")
except ImportError:
    print("Installing torch (must come before jax)...")
    subprocess.run(["pip", "install", "-q", "torch"],
                   check=True, capture_output=True, text=True)

# --- Install JAX + other Python dependencies ---
required_pkgs = ["jax", "jaxlib", "flax", "diffrax", "equinox", "matplotlib", "torchdiffeq", "scipy", "pysmt", "palettable"]
missing = []
for pkg in required_pkgs:
    try:
        importlib.import_module(pkg)
    except ImportError:
        missing.append(pkg)

if missing:
    print(f"Installing missing packages: {missing}")
    if "jax" in missing or "jaxlib" in missing:
        subprocess.run(["pip", "install", "-q", "jax[cuda12]"],
                       check=True, capture_output=True, text=True)
        missing = [p for p in missing if p not in ("jax", "jaxlib")]
    if missing:
        subprocess.run(["pip", "install", "-q"] + missing,
                       check=True, capture_output=True, text=True)
    print("Installation complete.")
else:
    print("All required packages already installed.")

# --- Clone and install Ark ---
try:
    import ark
    print("Ark already available.")
except ImportError:
    print("Ark not found. Cloning from WangYuNeng/Ark...")
    ark_clone_target = "/content/Ark"
    if not os.path.exists(ark_clone_target):
        subprocess.run(
            ["git", "clone", "https://github.com/WangYuNeng/Ark.git", ark_clone_target],
            check=True, capture_output=True, text=True
        )
    print("Installing Ark from source...")
    subprocess.run(
        ["pip", "install", "-q", "-e", "."],
        cwd=ark_clone_target, check=True, capture_output=True, text=True
    )
    if ark_clone_target not in sys.path:
        sys.path.insert(0, ark_clone_target)
    print("Ark installed.")

# --- Set checkpoint path ---
ckpt_root = os.path.join(repo_path, "experiments", "cross_arch_tolerance", "checkpoints")
assert os.path.isdir(ckpt_root), f"Checkpoints directory not found: {ckpt_root}"
print("Checkpoints path:", ckpt_root)

# --- Final imports ---
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import diffrax
from ark.optimization.base_module import TimeInfo

print("JAX devices:", jax.devices())
print("\nSUCCESS! Setup complete.")


## 2.1 Neural ODE (Time-Augmented MLP Vector Field)

**Physics:** `dz/dt = MLP([z, t]; theta)`

**Architecture:** 3-layer MLP with `tanh` activations, time concatenated as extra input feature.

**Plain-class rationale:** In a CDG, edges and their attributes are fixed at compile time. A time-varying input like `t` that gets concatenated into the state would require a dynamic production rule â€” outside Ark's current grammar. The plain class handles this in `ode_fn` directly.

A Neural ODE treats the hidden state as evolving continuously in time: `dz/dt = MLP([z, t])`. Unlike DEQ, there is no fixed point to converge to; the state at `t=1` is simply the output. What we are watching is how a single initial point `z0` gets pushed through the vector field defined by the trained MLP. The trajectory shape is the model's prediction.

**Why plain class:** In a CDG, every edge weight is fixed at compile time. The time scalar `t` is a runtime variable that gets concatenated into the state vector — there is no CDG production rule for 'append a changing scalar to the input at every timestep.' The plain class handles this directly in `ode_fn` by building `[z, t]` on the fly.

**Analog mapping:** Same crossbar MLP as any digital accelerator, but running in continuous time. Each layer is a resistive crossbar (W1, W2, W3) followed by a differential-pair tanh squasher. The time scalar `t` is an extra input voltage fed into the first crossbar alongside `z`. The difference from the CDG path is that the compiler does not generate these layers; we wire them by hand in Python, but the physical circuit is identical: three crossbars + two nonlinear stages.


In [ ]:
from experiments.cross_arch_tolerance.models.neural_ode import load_model as load_ode
from neuro_analog.revised_ark_bridge.plain_fallback.mlp_field import build_neural_ode

ode_pt = load_ode(os.path.join(ckpt_root, "neural_ode.pt"))
ckt_ode = build_neural_ode(ode_pt, mismatch_sigma=0.0)

print("Neural ODE circuit built.")
print("Weight keys:", list(ckt_ode._keys))
print("Solver:", ckt_ode.solver)

### MLP Field Visualization

The Neural ODE and Flow circuits use the same **MLPFieldCkt** class: a 3-layer MLP where time `t` is concatenated as an extra input feature. The schematic below shows the crossbar-array view: each layer is a column of neurons, and dense connections represent the weight matrix MVMs on the analog crossbar.


In [ ]:
# Self-contained MLP crossbar schematic (bigger figure, darker edges, nothing cut off)
w_dict = {k: np.array(ckt_ode._parse_args(ckt_ode.a_trainable)[k]) for k in ckt_ode._keys}
sizes  = [w_dict["W1"].shape[1], w_dict["W1"].shape[0], w_dict["W2"].shape[0], w_dict["W3"].shape[0]]  # [3,20,20,2]
labels = ["Input [z, t]", "Hidden 1 (tanh)", "Hidden 2 (tanh)", "Output dz/dt"]
colors = ["#F4C20D", "#34A853", "#34A853", "#4285F4"]
wlabels = [f"W1: {w_dict['W1'].shape}", f"W2: {w_dict['W2'].shape}", f"W3: {w_dict['W3'].shape}"]
xs = [0, 1, 2, 3]
ypos = lambda n: np.array([0.5]) if n == 1 else np.linspace(0.08, 0.92, n)

fig, ax = plt.subplots(figsize=(15, 8))
for li in range(3):
    for a in ypos(sizes[li]):
        for b in ypos(sizes[li + 1]):
            ax.plot([xs[li], xs[li + 1]], [a, b], color="#23304d", alpha=0.30, lw=0.6, zorder=1)
for x, n, c, lab in zip(xs, sizes, colors, labels):
    y = ypos(n)
    ax.scatter([x] * n, y, s=300, c=c, edgecolors="black", linewidths=1.0, zorder=3)
    ax.text(x, 1.05, lab, ha="center", va="bottom", fontsize=13, fontweight="bold")
for li in range(3):
    ax.text((xs[li] + xs[li + 1]) / 2, -0.09, wlabels[li], ha="center", va="top",
            fontsize=12, color="#23304d")
ax.set_xlim(-0.5, 3.5); ax.set_ylim(-0.20, 1.22); ax.axis("off")
ax.set_title("Neural ODE  MLPFieldCkt:  3-layer crossbar (time-augmented input [z, t])",
             fontsize=15, fontweight="bold", pad=26)
plt.tight_layout(); plt.show()


In [ ]:
# MLPFieldCkt.readout returns only the FINAL state z(t1) (the model's output).
# For visualization we want the whole path, so override readout to return the full trajectory.
from neuro_analog.revised_ark_bridge.plain_fallback.mlp_field import MLPFieldCkt
MLPFieldCkt.readout = lambda self, y: y

# NOMINAL TRAJECTORY: push z0 forward through the learned vector field
z0 = jnp.array([0.5, -0.3])
ti_ode = TimeInfo(t0=0.0, t1=1.0, dt0=0.01, saveat=jnp.linspace(0.0, 1.0, 50))
out_ode = ckt_ode(ti_ode, z0, switch=jnp.array([]), args_seed=0, noise_seed=0)
traj = np.array(out_ode)           # (50, 2)
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(traj[:, 0], traj[:, 1], 'o-', markersize=3, alpha=0.7, label="Nominal")
ax.scatter(traj[0, 0], traj[0, 1], color='green', s=50, zorder=5, label="z0")
ax.scatter(traj[-1, 0], traj[-1, 1], color='red', s=50, zorder=5, label="z1")
ax.set_xlabel("z[0]"); ax.set_ylabel("z[1]")
ax.set_title("Neural ODE Trajectory (Nominal)")
ax.legend(); ax.set_aspect('equal'); plt.tight_layout(); plt.show()
print(f"Final state: [{traj[-1, 0]:.3f}, {traj[-1, 1]:.3f}]")


In [ ]:
# MISMATCH SPREAD: larger sigma + more draws + endpoint scatter so the fan is visible
from neuro_analog.revised_ark_bridge.plain_fallback.mlp_field import build_neural_ode
ckt_ode_mis = build_neural_ode(ode_pt, mismatch_sigma=0.3)

fig, ax = plt.subplots(figsize=(5.5, 5.5))
finals = []
for seed in range(20):
    tj = np.array(ckt_ode_mis(ti_ode, z0, switch=jnp.array([]), args_seed=seed + 1, noise_seed=0))
    ax.plot(tj[:, 0], tj[:, 1], color='steelblue', alpha=0.35, lw=1)
    finals.append(tj[-1])
finals = np.array(finals)
ax.plot(traj[:, 0], traj[:, 1], 'o-', color='black', markersize=3, label="Nominal (sigma=0)")
ax.scatter(finals[:, 0], finals[:, 1], color='crimson', s=18, zorder=5, label="endpoints (sigma=0.3)")
ax.set_xlabel("z[0]"); ax.set_ylabel("z[1]")
ax.set_title("Neural ODE: trajectory spread under mismatch (sigma=0.3, 20 draws)")
ax.legend(); ax.set_aspect('equal'); plt.tight_layout(); plt.show()
print(f"Endpoint std: z0={finals[:,0].std():.3f}, z1={finals[:,1].std():.3f}")


## 2.2 Normalizing Flow (Same MLP, Different Weights)

**Physics:** Identical to Neural ODE â€” `dz/dt = MLP([z, t]; theta)`. Only the trained weights differ (trained via rectified flow matching, not density estimation).

The same `MLPFieldCkt` class covers multiple architectures. The bridge is parameterized by weight extraction, not by architecture name.

Normalizing Flow uses the exact same architecture as Neural ODE — a 3-layer MLP with time-augmented input. The difference is how it was trained. The Neural ODE was trained by minimizing a supervised loss on the final state. The Flow was trained via rectified flow matching: it learns to transport samples from a Gaussian prior to the data distribution along straight-line trajectories. Both use `dz/dt = MLP([z, t])`, but the weights encode different objective functions.

What we are looking for: the side-by-side trajectories show that identical circuit topology with different trained weights produces completely different dynamics. This confirms our bridge is parameterized by weight extraction, not by hard-coding architecture-specific logic. One circuit class, many models.

**Analog mapping:** Identical to Neural ODE — three crossbar layers + tanh squashers. The same physical chip can run either model just by programming different conductances into the memristor array.


In [ ]:
from experiments.cross_arch_tolerance.models.flow import load_model as load_flow
from neuro_analog.revised_ark_bridge.plain_fallback.mlp_field import build_flow

flow_pt = load_flow(os.path.join(ckpt_root, "flow.pt"))
ckt_flow = build_flow(flow_pt, mismatch_sigma=0.0)
z0_flow = jnp.array([0.5, -0.3])     # 1-D state
ti_flow = TimeInfo(t0=0.0, t1=1.0, dt0=0.01, saveat=jnp.linspace(0.0, 1.0, 50))
out_flow = ckt_flow(ti_flow, z0_flow, switch=jnp.array([]), args_seed=0, noise_seed=0)
traj_flow = np.array(out_flow)        # (50, 2)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].plot(traj[:, 0], traj[:, 1], 'o-', markersize=3, label="Neural ODE")
axes[0].scatter(traj[0, 0], traj[0, 1], color='green', s=40, zorder=5)
axes[0].scatter(traj[-1, 0], traj[-1, 1], color='red', s=40, zorder=5)
axes[0].set_title("Neural ODE"); axes[0].set_xlabel("z[0]"); axes[0].set_ylabel("z[1]")
axes[0].set_aspect('equal'); axes[0].legend()
axes[1].plot(traj_flow[:, 0], traj_flow[:, 1], 'o-', markersize=3, color='purple', label="Flow")
axes[1].scatter(traj_flow[0, 0], traj_flow[0, 1], color='green', s=40, zorder=5)
axes[1].scatter(traj_flow[-1, 0], traj_flow[-1, 1], color='red', s=40, zorder=5)
axes[1].set_title("Normalizing Flow"); axes[1].set_xlabel("z[0]"); axes[1].set_ylabel("z[1]")
axes[1].set_aspect('equal'); axes[1].legend()
plt.suptitle("Same MLPFieldCkt class, different trained weights")
plt.tight_layout(); plt.show()


## 2.3 Diffusion: Critically-Damped Langevin Dynamics (CLD) SDE

**Generative dynamics (2nd-order damped oscillator + thermal noise):**
```
dx/dt = (beta/M) * v
dv/dt = -beta*x - (Gamma*beta/M)*v - score_theta(x, t) + sqrt(2*Gamma*beta)*eta(t)
```

This is the **generative (sampling) SDE**, not the forward noising process. CLD augments each data coordinate `x` with a momentum coordinate `v`; together they form a coupled damped oscillator. The learned `score_theta` enters as a restoring force on the momentum, and Johnson-Nyquist thermal noise is injected on `v`. Running this SDE transports a simple equilibrium distribution into the data distribution - the analog circuit *is* the sampler. (The forward noising process is the same oscillator with the score term removed; we don't need it at inference.)

**Why this needs a plain class.** Four features break the static CDG grammar at once: (1) `is_stochastic=True` needs a diffrax `MultiTerm`/`ControlTerm` solver; (2) noise couples to `v` only, not `x`; (3) the `beta(t)` schedule is time-varying (discrete index into the trained schedule); (4) the score network is an MLP taking `x` plus a sinusoidal time embedding. None have CDG production-rule equivalents today.

**Why noise on `v` only.** In a physical Langevin system, thermal (Johnson-Nyquist) noise enters through the *dissipative* element - the damping on momentum - by the fluctuation-dissipation theorem. The damping term `-(Gamma*beta/M) v` and the noise `sqrt(2*Gamma*beta)` are two halves of the same physics, so the noise must sit on `v`, never directly on the position `x`. Getting this coupling right is exactly the detail a static grammar hides and a hand-written class makes explicit.

**Analog mapping.** Two coupled RC/LC resonators per pixel: one for position `x` (data), one for velocity `v` (momentum). The score network is a small crossbar MLP that reads `x` and drives a restoring force. Thermal noise is a current source on the velocity node only - the physical noise of the resistive damping element. `beta(t)` modulates the noise amplitude via a voltage-controlled source. We wire the 2nd-order coupling by hand because the compiler does not generate it.


In [ ]:
from experiments.cross_arch_tolerance.models.diffusion import load_model as load_diff
from neuro_analog.revised_ark_bridge.plain_fallback.diffusion import build_diffusion

diff_pt = load_diff(os.path.join(ckpt_root, "diffusion.pt"))
ckt_diff = build_diffusion(diff_pt, mismatch_sigma=0.0)

print("Diffusion CLD circuit built.")
print("is_stochastic:", ckt_diff.is_stochastic)
print("Score net keys:", list(ckt_diff._keys))
print("Beta schedule length:", ckt_diff._n_beta)

### CLD Circuit Visualization

The diffusion model uses a **Critically-Damped Langevin Dynamics (CLD)** circuit: a 2nd-order damped oscillator with thermal noise. The schematic shows the two coupled state variables (position `x` and velocity `v`), the score-network feedback, and the Johnson-Nyquist noise injection on the momentum state.


In [ ]:
import matplotlib.patches as mpatches
fig, ax = plt.subplots(figsize=(11, 5.2)); ax.set_xlim(0, 11); ax.set_ylim(0, 6); ax.axis("off")
def box(x, y, w, h, text, fc):
    ax.add_patch(mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.08", fc=fc, ec="black", lw=1.5))
    ax.text(x+w/2, y+h/2, text, ha="center", va="center", fontsize=12, fontweight="bold", color="white")
box(1.0, 2.6, 2.4, 1.0, "x  (position)", "#4285F4")
box(6.8, 2.6, 2.4, 1.0, "v  (momentum)", "#EA4335")
ax.annotate("", xy=(3.4, 3.35), xytext=(6.8, 3.35), arrowprops=dict(arrowstyle="-|>", color="#188038", lw=2.2))
ax.text(5.1, 3.6, "dx/dt = (β/M)·v", ha="center", color="#188038", fontsize=12)
ax.annotate("", xy=(6.8, 2.85), xytext=(3.4, 2.85), arrowprops=dict(arrowstyle="-|>", color="#4285F4", lw=2.2))
ax.text(5.1, 2.4, "−β·x   (restoring / spring)", ha="center", color="#4285F4", fontsize=12)
ax.annotate("", xy=(9.4, 2.7), xytext=(9.4, 3.5), arrowprops=dict(arrowstyle="-|>", color="#EA4335", lw=1.8, connectionstyle="arc3,rad=1.1"))
ax.text(10.0, 3.1, "−(Γβ/M)·v  (damp)", ha="left", va="center", color="#EA4335", fontsize=10)
ax.annotate("", xy=(8.0, 3.6), xytext=(8.0, 5.2), arrowprops=dict(arrowstyle="-|>", color="#9C27B0", lw=2, ls="--"))
ax.text(8.0, 5.45, "+√(2Γβ)·η(t)   (Johnson–Nyquist, on v only)", ha="center", color="#9C27B0", fontsize=11)
box(4.6, 0.4, 3.0, 0.9, "score_θ(x, t)  MLP", "#5F6368")
ax.annotate("", xy=(7.2, 2.6), xytext=(6.5, 1.3), arrowprops=dict(arrowstyle="-|>", color="#5F6368", lw=1.8))
ax.text(7.35, 1.85, "− score", color="#5F6368", fontsize=11)
ax.set_title("Diffusion CLD circuit: 2nd-order damped oscillator, thermal noise on momentum v", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()


In [ ]:
import warnings
from neuro_analog.revised_ark_bridge.plain_fallback.diffusion import CLDCkt
CLDCkt.readout = lambda self, y: y   # full (T,128) state for viz (default returns only x)

x0 = jnp.zeros(64); v0 = jnp.zeros(64)
state0 = jnp.concatenate([x0, v0])
ti_diff = TimeInfo(t0=0.0, t1=1.0, dt0=0.01, saveat=jnp.linspace(0.0, 1.0, 100))
with warnings.catch_warnings():
    warnings.simplefilter("ignore")   # Tsit5-on-SDE note (a dedicated SDE solver is the proper fix)
    out_diff = np.array(ckt_diff(ti_diff, state0, switch=jnp.array([]), args_seed=0, noise_seed=42))

t_axis = np.linspace(0.0, 1.0, out_diff.shape[0])
x_t, v_t = out_diff[:, :64], out_diff[:, 64:]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(t_axis, x_t[:, :6], alpha=0.85); axes[0].set_title("Position x(t) — integrates v (smooth)"); axes[0].set_xlabel("t"); axes[0].set_ylabel("x")
axes[1].plot(t_axis, v_t[:, :6], alpha=0.85); axes[1].set_title("Momentum v(t) — thermal noise injected here"); axes[1].set_xlabel("t"); axes[1].set_ylabel("v")
plt.suptitle("CLD generative SDE: one stochastic trajectory (noise enters v, propagates into x)")
plt.tight_layout(); plt.show()
print(f"Trajectory std:  x = {x_t.std():.4f}   v = {v_t.std():.4f}   (v > x confirms noise sits on momentum)")


In [ ]:
# NOISE-AMPLITUDE SCHEDULE: the thermal-noise gain on v is sqrt(2*Gamma*beta(t)).
# This is the schedule the circuit injects (illustrative - straight from the formula,
# with Gamma = 1.0; it is not a fit to simulation output).
betas = np.linspace(1e-4, 0.02, ckt_diff._n_beta)
noise_amps = np.sqrt(2.0 * 1.0 * betas)

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(betas, noise_amps, lw=2)
ax.set_xlabel("beta(t)  (noise schedule)")
ax.set_ylabel("Noise amplitude  sqrt(2*Gamma*beta)")
ax.set_title("Thermal-noise gain on the momentum state vs beta(t)")
plt.tight_layout()
plt.show()


## 2.4 Summary

| Family | Path | Why a plain class? | Key physics | Mismatch effect |
|--------|------|--------------------|-------------|-----------------|
| Neural ODE | Plain | Time-augmented MLP not expressible in a static CDG | `dz/dt = MLP([z,t])` | trajectory drift, density distortion |
| Flow | Plain | Same architecture as Neural ODE, different training | `dz/dt = MLP([z,t])` (rectified) | same as above |
| Diffusion | Plain | 2nd-order SDE, noise on v only, time-varying beta | CLD: `dx/dt=(beta/M)v`, `dv/dt= ... + sqrt(2*Gamma*beta)*eta` | score corruption -> generation collapse |

**Key takeaways:**
- The compiler is powerful but not universal. Rather than force-fit every architecture, we drop to hand-written `BaseAnalogCkt` subclasses when the physics is simpler to express directly.
- The contract is identical: `ode_fn`, `make_args`, `noise_fn`, `readout`. Mismatch-aware retraining through `jax.grad` still works because `a_trainable` is a flat array.
- The CLD detail that matters: Johnson-Nyquist noise belongs on the momentum state `v`, not the position `x` - a direct consequence of fluctuation-dissipation.
